In [1]:
# ============================================================
# INDICATORE 2 - DENSITA' RICETTIVA (comune x anno)
# Densita' ricettiva = letti totali / superficie comunale (letti/km2)
# Stessa convenzione della Densita' Turistica:
#   - RAW_DIR: file letti diretti da qui
#   - comune uniformato con .str.strip().str.title()
#
# PASSO 1: aggregazione letti per comune-anno + verifica su Cagliari
# ============================================================


from pathlib import Path
import pandas as pd
import duckdb

BASE_DIR = Path(r"../../")
RAW_DIR = BASE_DIR / "data" / "raw"

FILE_CAPACITA = RAW_DIR / "capacita_ricettiva.csv"

print("CSV utilizzato:")
print(f"  {FILE_CAPACITA}")

CSV utilizzato:
  ../../data/raw/capacita_ricettiva.csv


In [2]:
print(f"\n=== FILE: {FILE_CAPACITA.name} ===")
cap = pd.read_csv(FILE_CAPACITA)
cap["comune"] = cap["comune"].astype(str).str.strip().str.title()
n_anni = sorted(cap["anno"].unique())
n_comuni = cap["comune"].nunique()
print(f"righe sorgente: {len(cap)} | colonne: {list(cap.columns)}")
print(f"anni: {n_anni} | comuni distinti: {n_comuni} (su 377 attesi)")


=== FILE: capacita_ricettiva.csv ===
righe sorgente: 8589 | colonne: ['anno', 'provincia', 'comune', 'tipologia', 'categoria', 'numero_strutture', 'letti', 'camere']
anni: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)] | comuni distinti: 354 (su 377 attesi)


In [3]:
letti_annuali = (
    cap.groupby(["comune", "anno"], as_index=False)["letti"]
    .sum()
    .rename(columns={"letti": "letti_totali"})
)
print(f"\nDopo aggregazione (comune-anno): {len(letti_annuali)} righe")


Dopo aggregazione (comune-anno): 1351 righe


In [4]:
COMUNE_VERIFICA = "Cagliari"
r = letti_annuali[letti_annuali["comune"] == COMUNE_VERIFICA].sort_values("anno")
print(f"\n--- Verifica: '{COMUNE_VERIFICA}' ---")
print(r.to_string(index=False))

ultimo_anno = cap


--- Verifica: 'Cagliari' ---
  comune  anno  letti_totali
Cagliari  2022         10516
Cagliari  2023         11038
Cagliari  2024         14712
Cagliari  2025         17215


In [5]:
def chiave_comune(s):
    if pd.isna(s):
        return None
    apostrofo_tipografico = chr(0x2019)
    return str(s).strip().lower().replace(apostrofo_tipografico, "'")

ANNI = [2022, 2023, 2024, 2025]

FILE_SUPERFICIE = RAW_DIR / "superficie_comunale.csv"
superficie = pd.read_csv(FILE_SUPERFICIE)
superficie["comune"] = superficie["comune"].astype(str).str.strip().str.title()
superficie["chiave_comune"] = superficie["comune"].map(chiave_comune)

n_comuni_sup = superficie["comune"].nunique()
print(f"\n=== FILE: {FILE_SUPERFICIE.name} ===")
print(f"righe: {len(superficie)} | comuni: {n_comuni_sup}")


=== FILE: superficie_comunale.csv ===
righe: 377 | comuni: 377


In [6]:
cap["chiave_comune"] = cap["comune"].map(chiave_comune)
letti_annuali = (
    cap.groupby(["chiave_comune", "anno"], as_index=False)["letti"]
    .sum()
    .rename(columns={"letti": "letti_totali"})
)

In [7]:
# scaffold: tutti i 377 comuni x tutti gli anni
ANNI_DF = pd.DataFrame({"anno": ANNI})
scaffold = superficie[["chiave_comune", "comune", "superficie_kmq"]].merge(ANNI_DF, how="cross")

m = scaffold.merge(letti_annuali, on=["chiave_comune", "anno"], how="left")

# stesso nome/polarita' dell'Indicatore 1: True = dato osservato/affidabile
m["copertura_sufficiente"] = m["letti_totali"].notna()
m["letti_totali"] = m["letti_totali"].fillna(0).astype(int)
m["densita_ricettiva"] = (m["letti_totali"] / m["superficie_kmq"]).round(2)

print(m)

     chiave_comune   comune  superficie_kmq  anno  letti_totali  \
0           modolo   Modolo           2.616  2022            81   
1           modolo   Modolo           2.616  2023            82   
2           modolo   Modolo           2.616  2024            78   
3           modolo   Modolo           2.616  2025            96   
4          tinnura  Tinnura           3.827  2022             0   
...            ...      ...             ...   ...           ...   
1503         olbia    Olbia         385.439  2025         30267   
1504       sassari  Sassari         547.317  2022          2739   
1505       sassari  Sassari         547.317  2023          3217   
1506       sassari  Sassari         547.317  2024          3859   
1507       sassari  Sassari         547.317  2025          4725   

      copertura_sufficiente  densita_ricettiva  
0                      True              30.96  
1                      True              31.35  
2                      True              29.82  

In [8]:
# spalmo sui 12 mesi: valore REALE costante (l'offerta esiste tutto l'anno,
# non e' una stima statistica come la redistribuzione delle presenze)
MESI_DF = pd.DataFrame({"mese": range(1, 13)})
m_mensile = m.merge(MESI_DF, how="cross")

out_ricettiva = m_mensile[["comune", "anno", "mese", "letti_totali", "superficie_kmq",
                            "densita_ricettiva", "copertura_sufficiente"]].sort_values(["comune", "anno", "mese"])

n_attese = n_comuni_sup * len(ANNI) * 12
print(f"\nRighe totali: {len(out_ricettiva)} (attese: {n_comuni_sup} comuni x {len(ANNI)} anni x 12 mesi = {n_attese})")

for comune_verifica in ["Cagliari", "Bidonì"]:
    print(f"\n--- Verifica: '{comune_verifica}' (2025) ---")
    print(out_ricettiva[(out_ricettiva["comune"] == comune_verifica) &
                         (out_ricettiva["anno"] == 2025)].to_string(index=False))

n_ok = out_ricettiva["copertura_sufficiente"].sum()
print(f"\nRighe con copertura_sufficiente=True: {n_ok} su {len(out_ricettiva)}")


Righe totali: 18096 (attese: 377 comuni x 4 anni x 12 mesi = 18096)

--- Verifica: 'Cagliari' (2025) ---
  comune  anno  mese  letti_totali  superficie_kmq  densita_ricettiva  copertura_sufficiente
Cagliari  2025     1         17215          84.916             202.73                   True
Cagliari  2025     2         17215          84.916             202.73                   True
Cagliari  2025     3         17215          84.916             202.73                   True
Cagliari  2025     4         17215          84.916             202.73                   True
Cagliari  2025     5         17215          84.916             202.73                   True
Cagliari  2025     6         17215          84.916             202.73                   True
Cagliari  2025     7         17215          84.916             202.73                   True
Cagliari  2025     8         17215          84.916             202.73                   True
Cagliari  2025     9         17215          84.916       

In [ ]:
# spalmo sui 12 mesi: valore REALE costante (l'offerta esiste tutto l'anno,
# non e' una stima statistica come la redistribuzione delle presenze)
MESI_DF = pd.DataFrame({"mese": range(1, 13)})
m_mensile = m.merge(MESI_DF, how="cross")

out_ricettiva = m_mensile[["comune", "anno", "mese", "letti_totali", "superficie_kmq",
                            "densita_ricettiva", "copertura_sufficiente"]].sort_values(["comune", "anno", "mese"])

n_attese = n_comuni_sup * len(ANNI) * 12
print(f"\nRighe totali: {len(out_ricettiva)} (attese: {n_comuni_sup} comuni x {len(ANNI)} anni x 12 mesi = {n_attese})")

for comune_verifica in ["Cagliari", "Bidonì"]:
    print(f"\n--- Verifica: '{comune_verifica}' (2025) ---")
    print(out_ricettiva[(out_ricettiva["comune"] == comune_verifica) &
                         (out_ricettiva["anno"] == 2025)].to_string(index=False))

n_ok = out_ricettiva["copertura_sufficiente"].sum()
print(f"\nRighe con copertura_sufficiente=True: {n_ok} su {len(out_ricettiva)}")


Righe totali: 18096 (attese: 377 comuni x 4 anni x 12 mesi = 18096)

--- Verifica: 'Cagliari' (2025) ---
  comune  anno  mese  letti_totali  superficie_kmq  densita_ricettiva  copertura_sufficiente
Cagliari  2025     1         17215          84.916             202.73                   True
Cagliari  2025     2         17215          84.916             202.73                   True
Cagliari  2025     3         17215          84.916             202.73                   True
Cagliari  2025     4         17215          84.916             202.73                   True
Cagliari  2025     5         17215          84.916             202.73                   True
Cagliari  2025     6         17215          84.916             202.73                   True
Cagliari  2025     7         17215          84.916             202.73                   True
Cagliari  2025     8         17215          84.916             202.73                   True
Cagliari  2025     9         17215          84.916       

In [15]:
# ============================================================
# PASSO 3 - Salvataggio CSV per anno + scrittura nel database
# ============================================================
OUT_DIR = BASE_DIR / "data" / "indicatore_densita_ricettiva"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_DIR = BASE_DIR / "db"
DB_PATH = DB_DIR / "sardegna_overtourism.duckdb"
DB_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))
con.execute("CREATE SCHEMA IF NOT EXISTS presentation")

for anno in ANNI:
    out_anno = out_ricettiva[out_ricettiva["anno"] == anno].reset_index(drop=True)
    percorso_out = OUT_DIR / f"densita_ricettiva_{anno}.csv"
    out_anno.to_csv(percorso_out, index=False, encoding="utf-8-sig")
    print(f"{anno}: {out_anno['comune'].nunique()} comuni | {len(out_anno)} righe | salvato -> {percorso_out}")

out_ricettiva.to_csv("densita_ricettiva_complessiva.csv", index=False, encoding="utf-8-sig")

con.register("densita_ricettiva_temp", out_ricettiva)
con.execute("""
    CREATE OR REPLACE TABLE presentation.densita_ricettiva AS
    SELECT * FROM densita_ricettiva_temp
""")

verifica_db = con.execute("""
    SELECT COUNT(*) AS n_righe, COUNT(DISTINCT comune) AS n_comuni,
           COUNT(DISTINCT anno) AS n_anni
    FROM presentation.densita_ricettiva
""").df()
print("\n=== SCRITTURA NEL DATABASE - presentation.densita_ricettiva ===")
print(verifica_db.to_string(index=False))

print("\n=== VERIFICA DAL DB - Cagliari 2025 ===")
print(con.execute("""
    SELECT comune, anno, mese, letti_totali, densita_ricettiva
    FROM presentation.densita_ricettiva
    WHERE comune = 'Cagliari' AND anno = 2025
    ORDER BY mese
""").df().to_string(index=False))

2022: 377 comuni | 4524 righe | salvato -> ../../data/indicatore_densita_ricettiva/densita_ricettiva_2022.csv
2023: 377 comuni | 4524 righe | salvato -> ../../data/indicatore_densita_ricettiva/densita_ricettiva_2023.csv
2024: 377 comuni | 4524 righe | salvato -> ../../data/indicatore_densita_ricettiva/densita_ricettiva_2024.csv
2025: 377 comuni | 4524 righe | salvato -> ../../data/indicatore_densita_ricettiva/densita_ricettiva_2025.csv

=== SCRITTURA NEL DATABASE - presentation.densita_ricettiva ===
 n_righe  n_comuni  n_anni
   18096       377       4

=== VERIFICA DAL DB - Cagliari 2025 ===
  comune  anno  mese  letti_totali  densita_ricettiva
Cagliari  2025     1         17215             202.73
Cagliari  2025     2         17215             202.73
Cagliari  2025     3         17215             202.73
Cagliari  2025     4         17215             202.73
Cagliari  2025     5         17215             202.73
Cagliari  2025     6         17215             202.73
Cagliari  2025     7  

In [16]:
con.close()
print("Connessione al database chiusa.")

Connessione al database chiusa.
